# Arquitectura del Modelo: MasterStockTransformer

Este notebook toma los datos procesados en el paso anterior y define la estructura profunda del modelo en PyTorch. Convertiremos los datos tabulares en secuencias temporales (ventanas móviles de 10 días) y diseñaremos el mecanismo de Cross-Attention propuesto.

In [1]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(42)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Configurar el dispositivo (GPU si está disponible, sino CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: mps


### 1. Creación del Dataset de PyTorch (Sliding Windows)

Las redes neuronales recurrentes y Transformers necesitan que los datos tengan la forma `(Batch_Size, Sequence_Length, Features)`.
Usaremos una ventana de `seq_length = 10` días para predecir el día `T+1`.

In [2]:
class StockDataset(Dataset):
    def __init__(self, data, target_col_idx, context_col_indices, seq_length=10):
        """
        data: numpy array con los datos escalados.
        target_col_idx: índice de la columna que queremos predecir (ej. JPM_Log_Ret).
        context_col_indices: índices de las columnas que sirven de contexto (el resto de los bancos y XLF).
        seq_length: número de días pasados a observar (10 días).
        """
        self.data = data
        self.target_col_idx = target_col_idx
        self.context_col_indices = context_col_indices
        self.seq_length = seq_length

    def __len__(self):
        # Retornamos la cantidad de secuencias posibles
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        # Extraer la ventana de tiempo para X (desde idx hasta idx + seq_length)
        window = self.data[idx : idx + self.seq_length]
        
        # El objetivo (Target - JPM) y el contexto (Mercado) en el pasado
        x_target = window[:, self.target_col_idx]  # Forma: (seq_length, features_target)
        x_context = window[:, self.context_col_indices] # Forma: (seq_length, features_context)
        
        # Y es el valor futuro que queremos predecir (en el día seq_length + 1)
        y = self.data[idx + self.seq_length, self.target_col_idx[0]] # El retorno de JPM de "mañana"
        
        return torch.tensor(x_target, dtype=torch.float32), \
               torch.tensor(x_context, dtype=torch.float32), \
               torch.tensor(y, dtype=torch.float32)

print("Clase StockDataset definida correctamente.")

Clase StockDataset definida correctamente.


### 2. Arquitectura: MasterStockTransformer

El núcleo inspirado en el paper MASTER. Usaremos la historia de JPM como **Query**, y la historia del mercado general como **Key / Value** dentro de un bloque de `MultiheadAttention`.

In [3]:
class MasterStockTransformer(nn.Module):
    def __init__(self, target_features, context_features, d_model=64, nhead=4, num_layers=1, dropout=0.1):
        super(MasterStockTransformer, self).__init__()
        
        # 1. Feature Projection (Mapeo de las variables crudas a la dimensión oculta d_model)
        self.target_proj = nn.Linear(target_features, d_model)
        self.context_proj = nn.Linear(context_features, d_model)
        
        # 2. Mecanismo de Cross-Attention
        # Query: Target (JPM), Key/Value: Contexto (Resto del mercado)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)
        
        # 3. FeedForward Network (FFN)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model)
        )
        
        # Normalizaciones de capa
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # 4. Capa de Predicción Final
        self.predictor = nn.Linear(d_model, 1)
        
    def forward(self, x_target, x_context):
        """
        x_target: (Batch, Seq_Length, Target_Features)
        x_context: (Batch, Seq_Length, Context_Features)
        """
        # Proyectar al espacio d_model
        q = self.target_proj(x_target)   # Query: El banco a predecir (JPM)
        k = self.context_proj(x_context) # Key: El entorno macro y competencia
        v = k                            # Value: Igual al Key
        
        # Aplicar Cross-Attention
        # attn_output shape: (Batch, Seq_Length, d_model)
        attn_output, attn_weights = self.cross_attention(query=q, key=k, value=v)
        
        # Conexión residual y Normalización
        x = self.norm1(q + attn_output)
        
        # FFN + Conexión residual y Normalización
        ffn_output = self.ffn(x)
        x = self.norm2(x + ffn_output)
        
        # Para la predicción, tomaremos la salida del último día de la secuencia (el tiempo T)
        last_timestep_output = x[:, -1, :] # Forma: (Batch, d_model)
        
        # Predicción final
        prediction = self.predictor(last_timestep_output)
        return prediction.squeeze(-1) # Forma: (Batch,)

print("Arquitectura MasterStockTransformer construida con éxito.")

Arquitectura MasterStockTransformer construida con éxito.


### 3. Prueba de Verificación (Sanity Check)
Validaremos que las dimensiones matemáticas cuadren generando datos falsos (dummy data) con las dimensiones de nuestro problema real.

In [4]:
# Suponiendo: 
# 1 característica para el target (ej. Retorno de JPM)
# 4 características de contexto (ej. Retornos de BAC, C, WFC, XLF)
target_feats = 1
context_feats = 4
seq_len = 10
batch_size = 32

# Instanciar el modelo
model = MasterStockTransformer(target_features=target_feats, context_features=context_feats).to(device)

# Crear tensores falsos (Dummy Data)
dummy_target = torch.randn(batch_size, seq_len, target_feats).to(device)
dummy_context = torch.randn(batch_size, seq_len, context_feats).to(device)

# Forward pass
output = model(dummy_target, dummy_context)

print(f"Forma del input Target: {dummy_target.shape}")
print(f"Forma del input Contexto: {dummy_context.shape}")
print(f"Forma de la Predicción (Output): {output.shape}")

if output.shape == torch.Size([batch_size]):
    print("¡Prueba superada! Las dimensiones matemáticas fluyen correctamente.")
else:
    print("Hubo un problema con las dimensiones de salida.")

Forma del input Target: torch.Size([32, 10, 1])
Forma del input Contexto: torch.Size([32, 10, 4])
Forma de la Predicción (Output): torch.Size([32])
¡Prueba superada! Las dimensiones matemáticas fluyen correctamente.
